# Fairness Audit Demo - FairML Consulting
This notebook demonstrates the capabilities of the **Measurement Module** for auditing bias in loan approval datasets.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analyzer import FairnessAnalyzer, FairnessResult

# Setting plot style for better visualizations later
plt.style.use('ggplot')
%matplotlib inline

# Load the dataset
file_path = 'Loan Dataset.csv'
df_raw = pd.read_csv(file_path)

# Clean column names just in case there are leading/trailing spaces
df_raw.columns = df_raw.columns.str.strip()

# Display the first few rows and column info to verify the schema
print(f"Dataset shape: {df_raw.shape}")
display(df_raw.head())
print(df_raw.info())

print("Setup complete. FairnessAnalyzer imported.")

Dataset shape: (52000, 27)


,Applicant_ID,Gender,Age,Marital_Status,Dependents,Education,Employment_Status,Occupation_Type,Residential_Status,City/Town,...,Loan_Amount_Requested,Loan_Term,Loan_Purpose,Interest_Rate,Loan_Type,Co-Applicant,Bank_Account_History,Transaction_Frequency,Default_Risk,Loan_Approval_Status
0,1,Female,25,Married,2,Graduate,Employed,Business,Own,Urban,...,24535,209,Home,4.27,Secured,Yes,8,20,0.81,1
1,2,Male,36,Married,2,High School,Employed,Business,Own,Suburban,...,8288,33,Home,14.78,Unsecured,Yes,9,9,0.17,0
2,3,Male,43,Single,0,Postgraduate,Self-Employed,Freelancer,Own,Urban,...,10308,159,Vehicle,12.33,Secured,Yes,7,27,0.25,0
3,4,Female,28,Married,0,High School,Self-Employed,Freelancer,Rent,Suburban,...,33937,39,Personal,8.77,Secured,No,9,16,0.27,1
4,5,Female,32,Single,0,Graduate,Employed,Salaried,Rent,Suburban,...,23360,34,Home,9.04,Unsecured,No,1,17,0.32,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52000 entries, 0 to 51999
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Applicant_ID                52000 non-null  int64  
 1   Gender                      52000 non-null  object 
 2   Age                         52000 non-null  int64  
 3   Marital_Status              52000 non-null  object 
 4   Dependents                  52000 non-null  int64  
 5   Education                   52000 non-null  object 
 6   Employment_Status           52000 non-null  object 
 7   Occupation_Type             52000 non-null  object 
 8   Residential_Status          52000 non-null  object 
 9   City/Town                   52000 non-null  object 
 10  Annual_Income               52000 non-null  int64  
 11  Monthly_Expenses            52000 non-null  int64  
 12  Credit_Score                52000 non-null  int64  
 13  Existing_Loans              520

In [2]:
# Check unique values of the target to identify the positive label
print("Target column values:", df_raw['Loan_Approval_Status'].unique())

# Initialize the analyzer
# Note: Adjust positive_label if your dataset uses 'Approved' or 1
analyzer = FairnessAnalyzer(
    df=df_raw,
    target_col='Loan_Approval_Status',
    sensitive_col='Gender',
    positive_label=1 
)

# Alternative fluent configuration
analyzer.set_config(
    target_col='Loan_Approval_Status',
    sensitive_col='Gender',
    positive_label=1
)

Target column values: [1 0]
FairnessAnalyzer ready — 52,000 rows loaded.


Binning the age into grops so that we can have more significant metrics 

In [3]:
# Define bins and labels
age_bins = [0, 25, 35, 45, 55, 65, np.inf]
age_labels = ['0-25', '26-35', '36-45', '46-55', '56-65', '65+']

# Execute binning
# We set set_as_sensitive=False for now to keep 'Gender' as the primary focus,
# but the method allows switching it automatically.
analyzer.bin_column(
    column_name='Age',
    bins=age_bins,
    labels=age_labels,
    new_column_name='Age_Group',
    set_as_sensitive=False
)

# Verify the result
print("\nDistribution of the new 'Age_Group' column:")
print(analyzer.df['Age_Group'].value_counts().sort_index())

display(analyzer.df[['Age', 'Age_Group']].head(10))

'Age' → 'Age_Group' (set_as_sensitive=False).

Distribution of the new 'Age_Group' column:
Age_Group
0-25      3889
26-35    17448
36-45    17835
46-55     8716
56-65     2966
65+       1146
Name: count, dtype: int64


,Age,Age_Group
0,25,0-25
1,36,36-45
2,43,36-45
3,28,26-35
4,32,26-35
5,49,46-55
6,41,36-45
7,52,46-55
8,27,26-35
9,61,56-65


In [4]:
analyzer.sensitive_col = 'Age_Group'

print(f"--- Analyzing Fairness for: {analyzer.sensitive_col} ---")

# Calculate metrics for the age groups
age_metrics = analyzer.calculate_classification_metrics(
    y_pred=None,
    engine="fairlearn",
    min_group_size=10, # Filtering out groups with very few samples
    n_bootstrap=50
)

for name, res in age_metrics.items():
    print(res)
    # Access metadata to see group-specific rates
    print(f"Positive rates per group: {res.metadata['positive_rate_by_group']}\n")

--- Analyzing Fairness for: Age_Group ---


C:\Users\glocc\AppData\Local\Temp\ipykernel_19688\1557387559.py:6: UserWarning: y_pred not provided – using target column as proxy. Metrics will reflect label disparities, not model bias.
  age_metrics = analyzer.calculate_classification_metrics(
c:\Users\glocc\OneDrive\Documenti\Visual Studio 2017\AI Ethics Final Project\analyzer.py:301: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = df.groupby(self.sensitive_col, sort=False)
c:\Users\glocc\OneDrive\Documenti\Visual Studio 2017\AI Ethics Final Project\analyzer.py:301: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = df.groupby(self.sensitive_co

FairnessResult(Demographic Parity Difference)
  value              = 0.5882
  95% CI             = (0.5688, 0.6085)
  effect_size        = 0.1956987935072432
  sample_sizes       = {'36-45': 17835, '26-35': 17448, '46-55': 8716, '0-25': 3889, '56-65': 2966, '65+': 1146}

Positive rates per group: {'0-25': 0.4193880174852147, '26-35': 0.7303988995873453, '36-45': 0.7312587608634707, '46-55': 0.6092244148692061, '56-65': 0.16014834794335805, '65+': 0.1431064572425829}

FairnessResult(Equalized Odds Difference)
  value              = 0.0000
  95% CI             = (0.0000, 0.0000)
  effect_size        = None
  sample_sizes       = {'36-45': 17835, '26-35': 17448, '46-55': 8716, '0-25': 3889, '56-65': 2966, '65+': 1146}

Positive rates per group: {'0-25': 0.4193880174852147, '26-35': 0.7303988995873453, '36-45': 0.7312587608634707, '46-55': 0.6092244148692061, '56-65': 0.16014834794335805, '65+': 0.1431064572425829}



### Ethical Assessment
The fairness audit reveals a **significant Disparate Impact** against younger and older age groups:
* **Primary Bias:** Applicants aged **56-65** and **65+** show approval rates below 16%, compared to the 73% of the **26-45** group.
* **Metric:** The Demographic Parity Ratio is well below the **0.8** industry standard, indicating that the loan approval process is not fair regarding the applicant's age.
* **Conclusion:** The model requires mitigation to reduce the bias against elderly applicants.

Obtaining metrics for other sensible attributes it's easy! Just change the `analyzer.sensitive_col`

In [5]:
# Running a deep statistical audit on Gender
analyzer.sensitive_col = 'Gender'

audit_result = analyzer.get_fairness_audit(n_bootstrap=200, random_state=42)

print("--- Statistical Fairness Audit (Gender) ---")
print(audit_result)

# Detailed breakdown from metadata
meta = audit_result.metadata
print(f"Privileged Group: {meta['privileged_group']} (Rate: {meta['privileged_rate']:.2%})")
print(f"Unprivileged Group: {meta['unprivileged_group']} (Rate: {meta['unprivileged_rate']:.2%})")
print(f"Risk Ratio (Effect Size): {audit_result.effect_size:.4f}")

--- Statistical Fairness Audit (Gender) ---
FairnessResult(Selection Rate Disparity)
  value              = 0.0040
  95% CI             = (-0.0031, 0.0120)
  effect_size        = 0.9937785536822049
  sample_sizes       = {'Male': 26011, 'Female': 25989}

Privileged Group: Female (Rate: 64.37%)
Unprivileged Group: Male (Rate: 63.97%)
Risk Ratio (Effect Size): 0.9938


#### Gender disparities
The Confidence Interval contains zero so there is no discrimination based on gender alone. The effect size being 99% means that the "unprivileged group" (in this case *Male*, despite not being significant) obtains 99% of the opportunites *Females* get.

In [7]:
# Perform intersectional analysis across Gender and Age_Group
# This will create combinations like 'Male_26-35', 'Female_0-25', etc.
intersectional_results = analyzer.intersectional_audit(
    column_names=['Gender', 'Age_Group'],
    intersectional_col='Gender_Age_Intersection',
    min_group_size=10,    # Only analyze groups with enough data
    n_bootstrap=50,
    apply_fdr_correction=True
)

print("--- Intersectional Fairness Audit ---")
# Filter to show interesting columns including the BH correction results
cols_view = [
    'group', 'n', 'selection_rate', 'ci_lower', 'ci_upper', 
    'significant_after_correction'
]
display(intersectional_results[cols_view].sort_values(by='selection_rate'))

# Highlight groups that show statistically significant disparity
sig_groups = intersectional_results[intersectional_results['significant_after_correction'] == True]
if not sig_groups.empty:
    print(f"\nFound {len(sig_groups)} groups with statistically significant disparities after FDR correction.")
else:
    print("\nNo groups showed statistically significant disparity after correction.")

--- Intersectional Fairness Audit ---


,group,n,selection_rate,ci_lower,ci_upper,significant_after_correction
10,Male_65+,589,0.120543,0.090266,0.143763,False
9,Male_56-65,1479,0.158215,0.139628,0.184799,False
8,Female_56-65,1487,0.162071,0.141972,0.182616,False
11,Female_65+,557,0.166966,0.137964,0.198732,False
6,Male_0-25,1987,0.412179,0.394172,0.427773,False
7,Female_0-25,1902,0.426919,0.409440,0.449107,False
5,Male_46-55,4344,0.607965,0.595655,0.626662,False
4,Female_46-55,4372,0.610476,0.601491,0.623232,False
3,Male_26-35,8665,0.727986,0.718501,0.735437,False
1,Female_36-45,8888,0.728735,0.717912,0.737177,False



No groups showed statistically significant disparity after correction.


Age is the dominant predictor of the selection rate. Gender does not introduce any additional disparity once age is controlled for. Older groups (56+) are systematically disadvantaged regardless of gender.

In [ ]:
analyzer.sensitive_col = 'Gender'

print(f"--- Analyzing Fairness for: {analyzer.sensitive_col} ---")

# Calculate metrics using Fairlearn engine
# We use the ground truth labels as predictions to see existing bias in the data
classification_results = analyzer.calculate_classification_metrics(
    y_pred=None, 
    engine="fairlearn",
    n_bootstrap=500, # Lowered iterations for speed in testing
    random_state=42
)

# Display results
for metric_name, result in classification_results.items():
    print(result)

# Generate and save the visualization
output_image = "gender_fairness_report.png"
analyzer.generate_report_visualizations(
    results=classification_results,
    output_path=output_image
)

# Display the generated image in the notebook
from IPython.display import Image
Image(filename=output_image)

--- Analyzing Fairness for: Gender ---


C:\Users\glocc\AppData\Local\Temp\ipykernel_19688\2220973552.py:7: UserWarning: y_pred not provided – using target column as proxy. Metrics will reflect label disparities, not model bias.
  classification_results = analyzer.calculate_classification_metrics(


In [ ]:
# 1. Testing the CI/CD assertion helper
# Let's check if the Demographic Parity Difference is within an acceptable range
try:
    dp_result = classification_results['demographic_parity_difference']
    # We assert that the absolute difference is less than 0.1 (10%)
    analyzer.assert_fairness(dp_result, threshold=0.1, metric="value")
    print("✓ CI/CD Check: Fairness threshold maintained.")
except AssertionError as e:
    print(f"⚠ CI/CD Check: {e}")

# 2. MLflow Logging (Simulated)
# If you have MLflow installed and a server running, you can uncomment the following:
"""
import mlflow

with mlflow.start_run(run_name="Fairness_Audit_v1"):
    analyzer.log_to_mlflow(classification_results)
    mlflow.log_artifact(output_image)
    print("Metrics and plot successfully logged to MLflow.")
"""